In [2]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

data_dir = Path("data") / 'raw'
gis_dir = data_dir / 'gis'
assessor_dir = data_dir / 'assessor'

# 1. Load the spatial files
taxlots = gpd.read_file(gis_dir /"TaxlotsPublic.shp")
cities = gpd.read_file(gis_dir /"Cities.shp")
situs = gpd.read_file(gis_dir /"Situs.shp")

# Inspect Taxlots
print(f"Taxlots shape: {taxlots.shape}")
print(f"Taxlots CRS: {taxlots.crs}")
taxlots.head(3)

Taxlots shape: (196024, 78)
Taxlots CRS: PROJCS["NAD83 / Washington South (ftUS)",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",45.3333333333333],PARAMETER["central_meridian",-120.5],PARAMETER["standard_parallel_1",45.8333333333333],PARAMETER["standard_parallel_2",47.3333333333333],PARAMETER["false_easting",1640416.66666667],PARAMETER["false_northing",0],UNIT["US survey foot",0.304800609601219,AUTHORITY["EPSG","9003"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


,Download,prop_id,LandProp_i,AcctTypeDe,AvYear,TaxStat,CurrentUse,TaxCode,Pt1,Pt1Desc,...,SaleExcise,PlatName,PlatCode,PlatType,PlatDate,CityUGA,PublishDt,Shape_Leng,Shape_Area,geometry
0,2026-08-07,6000,6000,Real,2025,T,None,37000,202,STORAGE WAREHOUSE,...,509929,None,None,None,NaT,Vancouver City,2026-08-09,3180.876554,297371.540681,"POLYGON ((1080838.641 121026.169, 1080716.785 ..."
1,2026-08-07,7000,7000,Real,2025,T,None,37000,201,DISTRIBUTION WAREHOUSE,...,523276,None,None,None,NaT,Vancouver City,2026-08-09,333.423744,1484.894954,"POLYGON ((1080863.272 120651.934, 1080844.337 ..."
2,2026-08-07,8000,8000,Real,2025,EX,None,37000,991,UNUSED OR VACANT LAND - NO IMPROVEMENTS,...,635727,None,None,None,NaT,Vancouver City,2026-08-09,1841.637788,144629.174188,"POLYGON ((1080577.161 122959.403, 1080763.838 ..."


In [3]:
# Print column names for each dataset to identify matching key columns
print("Taxlots columns:", taxlots.columns.tolist()[:10])
print("\nSitus columns:", situs.columns.tolist()[:10])

Taxlots columns: ['Download', 'prop_id', 'LandProp_i', 'AcctTypeDe', 'AvYear', 'TaxStat', 'CurrentUse', 'TaxCode', 'Pt1', 'Pt1Desc']

Situs columns: ['SitusId', 'SN', 'HsNbr', 'HsSub', 'StDir', 'StName', 'Stype', 'Building', 'Apt', 'Zp1']


In [4]:
# Load first 5 rows of Assessor data using pipe delimiter
value_df_sample = pd.read_csv("data/raw/Assessor/ValueMktTaxVw.txt", sep='|', nrows=5)
land_df_sample = pd.read_csv("data/raw/Assessor/PropLandVw.txt", sep='|', nrows=5)

print("ValueMktTaxVw Columns:")
print(value_df_sample.columns.tolist())

print("\nPropLandVw Columns:")
print(land_df_sample.columns.tolist())

ValueMktTaxVw Columns:
['prop_id', 'Prop_val_yr', 'sup_num', 'MarketVal', 'MktBldgVal', 'MktLandVal', 'TaxTotalValue']

PropLandVw Columns:
['Prop_id', 'AssrAc', 'AssrSqFt', 'STATUS', 'PT1', 'PT1Desc', 'PT2', 'PT2Desc', 'Cycle', 'Nbrhd', 'Z1', 'Z2', 'DOR_Use_Code']


In [7]:
# Assume 'SERIAL_NUM' is the key column (update name based on Step 2 & 3 findings)
id_col = 'prop_id' 

# Standardize IDs to clean strings (strip spaces/dashes)
taxlot_ids = taxlots[id_col].astype(str).str.strip()

# Read full ID column from Assessor file
assessor_ids = pd.read_csv("data/raw/Assessor/ValueMktTaxVw.txt", sep='|', usecols=[id_col])[id_col].astype(str).str.strip()

# Calculate match rate
matches = taxlot_ids.isin(assessor_ids).sum()
total_taxlots = len(taxlot_ids)
print(f"Match Rate: {matches} / {total_taxlots} ({matches / total_taxlots * 100:.2f}%)")

Match Rate: 196024 / 196024 (100.00%)


In [8]:
# 1. Ensure both layers share the exact same CRS
if taxlots.crs != camas_bnd.crs:
    camas_bnd = camas_bnd.to_crs(taxlots.crs)

# 2. Perform spatial join/intersection to find taxlots inside Camas
camas_taxlots = gpd.sjoin(taxlots, camas_bnd, predicate='intersects')

print(f"Total Clark County taxlots: {len(taxlots)}")
print(f"Camas taxlots: {len(camas_taxlots)}")

# 3. Quick plot check
ax = camas_taxlots.plot(figsize=(8, 8), color='teal', edgecolor='black', linewidth=0.2)
ax.set_title("Camas Taxlots (Spatial Filter Check)")

NameError: name 'camas_bnd' is not defined

In [6]:

# 1. Load Camas Taxlots
taxlots = gpd.read_file("data/raw/GIS/TaxlotsPublic.shp")
camas_taxlots = taxlots[taxlots['City'].str.contains('Camas', case=False, na=False)].copy()

# Ensure prop_id is string and stripped of whitespace
camas_taxlots['prop_id'] = camas_taxlots['prop_id'].astype(str).str.strip()

# Calculate GIS acres from Shape_Area
camas_taxlots['gis_acres'] = camas_taxlots['Shape_Area'] / 43560.0

# 2. Load and Filter Assessor Financial Data
value_df = pd.read_csv("data/raw/Assessor/ValueMktTaxVw.txt", sep='|', low_memory=False)
value_df['prop_id'] = value_df['prop_id'].astype(str).str.strip()

# Keep latest tax/appraisal year
latest_year = value_df['Prop_val_yr'].max()
value_latest = value_df[value_df['Prop_val_yr'] == latest_year].copy()

# 3. Merge Geometries with Assessor Values
merged = camas_taxlots.merge(
    value_latest[['prop_id', 'MarketVal', 'MktBldgVal', 'MktLandVal']], 
    on='prop_id', 
    how='left'
)

# 4. Calculate Value Per Acre
# Avoid zero-division errors for unmeasured or tiny parcels
merged['value_per_acre'] = merged.apply(
    lambda r: r['MarketVal'] / r['gis_acres'] if r['gis_acres'] > 0 else 0, 
    axis=1
)

# 5. Format Address from Situs Points
situs = gpd.read_file("data/raw/GIS/Situs.shp")
situs['prop_id'] = situs['SN'].astype(str).str.strip()  # SN maps to prop_id in Situs layer

# Construct clean address string
situs['full_address'] = (
    situs['HsNbr'].fillna('').astype(str) + ' ' +
    situs['StDir'].fillna('') + ' ' +
    situs['StName'].fillna('') + ' ' +
    situs['Stype'].fillna('') + ', Camas, WA ' +
    situs['Zp1'].fillna('').astype(str)
).str.replace(r'\s+', ' ', regex=True).str.strip()

# Drop duplicate addresses per prop_id if multiple points exist
situs_clean = situs.drop_duplicates(subset=['prop_id'])[['prop_id', 'full_address']]

# Join Address to Main Dataset
final_df = merged.merge(situs_clean, on='prop_id', how='left')

# Print Audit Results
print(f"Total Camas Parcels: {len(camas_taxlots)}")
print(f"Parcels with Financial Values Joined: {final_df['MarketVal'].notna().sum()}")
print(f"Parcels with Addresses Joined: {final_df['full_address'].notna().sum()}")
print("\nSample Output:")
final_df[['prop_id', 'full_address', 'MarketVal', 'gis_acres', 'value_per_acre']].head()

KeyError: 'City'